<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v2/blob/main/myStocks_Portfolio_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
!pip install pandas==2.2.2 -qq
!pip install pyxirr -qq
!pip install ta -qq
!pip install yfinance==0.2.59 -qq
'''

'\n!pip install pandas==2.2.2 -qq\n!pip install pyxirr -qq\n!pip install ta -qq\n!pip install yfinance==0.2.59 -qq\n'

In [2]:
# import libraries

import numpy as np
import pandas as pd
import datetime
import warnings
warnings.filterwarnings('ignore')
import yfinance as yfin
import ta
import matplotlib.pyplot as plt
import requests

In [3]:
# notebook start time
import time
import datetime
import pytz

start_time = time.time()

# function to print date timestamp
def print_date_time():
  tz_NY = pytz.timezone('Asia/Kolkata')
  datetime_NY = datetime.datetime.now(tz_NY)
  print("Run date time (IST):", datetime_NY.strftime("%Y-%m-%d %H:%M:%S"))

In [4]:
# function to get booked and reserved amount
def get_amt():
  # fetch the JSON data from the URL
  url = "https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioAmts.json"
  response = requests.get(url)
  pf_amts = response.json()  # parse the JSON data

  # extract values from the JSON
  py_booked_amt_dm = pf_amts["py_booked_amt_dm"]
  py_booked_amt_sv = pf_amts["py_booked_amt_sv"]
  cy_booked_amt_dm = pf_amts["cy_booked_amt_dm"]
  cy_booked_amt_sv = pf_amts["cy_booked_amt_sv"]
  reserve_amt_dm = pf_amts["reserve_amt_dm"]
  reserve_amt_sv = pf_amts["reserve_amt_sv"]

  # perform calculations
  py_booked_amt = py_booked_amt_dm + py_booked_amt_sv
  cy_booked_amt = cy_booked_amt_dm + cy_booked_amt_sv
  reserve_amt = reserve_amt_dm + reserve_amt_sv
  total_booked_amt = py_booked_amt + cy_booked_amt

  return total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt

gen_output = 0

In [5]:
# function to format the amount
def format_amt(number):
    abs_number = abs(number)

    if abs_number >= 1_00_00_000:  # Crores
        formatted_amt = f"{number / 1_00_00_000:.2f} C"
    elif abs_number >= 1_00_000:  # Lakhs
        formatted_amt = f"{number / 1_00_000:.2f} L"
    elif abs_number >= 1_000:  # Thousands
        formatted_amt = f"{number / 1_000:.2f} K"
    else:
        formatted_amt = f"{number:.2f}"

    return formatted_amt

# function to set start and end date
def get_start_end_date():
  start_date = (datetime.date.today() + datetime.timedelta(days=-365)).strftime('%Y-%m-%d')
  end_date = (datetime.date.today() + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
  return start_date, end_date

# function to fetch my portfolio csv
def get_mypfs_df():
  mypfs_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioStocks.csv')
  return mypfs_df

# function to fetch my prospects csv
def get_mypps_df():
  mypps_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myProspectsScrips.csv')
  return mypps_df

# function to fetch screener data
def get_myscreen_df():
  myscreen_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myScreenerDB.csv')
  return myscreen_df

# function to fetch momentum data
def get_myinvmt_df():
  myinvmt_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myInvestments.csv')
  return myinvmt_df

# function to get the stock ids
def get_stock_ids(df_pf):
  stock_n100 = df_pf['Symbol'].unique()

  exclude = ['CADILAHC','MMTC', 'MASFIN']

  stock_ids = df_pf[~df_pf['Symbol'].isin(exclude) ]['Symbol'].unique()

  #mypf = mypf[mypf.Forecast.notnull()]
  #stock_ids = mypf['Symbol'].unique()

  #stock_ids.sort()
  return stock_ids

# get features from screener data
def get_screener_features():
  myscreen_df = get_myscreen_df()
  cols = ['Symbol', 'EPS', 'MedPE', 'ROCE%', 'ROE%', 'CapType']
  return myscreen_df[cols]

# get relative strength
def get_relative_strength(stock_list):

    # Dictionary to store stock tickers and their corresponding percentage price change
    stock_changes = {}

    # Loop through each stock and fetch its price data
    for stock in stock_list:
        try:
            # Download the stock data for the given date range
            stock_data = yfin.Ticker(stock + '.NS').history(period='1mo', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

            # Calculate the percentage change for the stock
            if len(stock_data) > 0:
                start_price = stock_data['Close'].iloc[0]
                end_price = stock_data['Close'].iloc[-1]
                percent_change = round(((end_price - start_price) / start_price) * 100, 2)
                stock_changes[stock] = percent_change
            else:
                stock_changes[stock] = np.nan  # If no data is available, set to NaN

        except Exception as e:
            print(f"Error fetching data for {stock}: {e}")
            stock_changes[stock] = np.nan

    # Create a DataFrame with stock tickers and their percentage changes
    df = pd.DataFrame(list(stock_changes.items()), columns=['Symbol', 'Percent_Change'])

    # Drop any stocks with missing data (NaN values)
    df = df.dropna()

    # Calculate the percentile rank based on percentage change
    df['RSP'] = round(df['Percent_Change'].rank(pct=True) * 100, 2)
    cols = ['Symbol', 'RSP']
    df = df[cols]
    # Sort by percentile rank (optional)
    df = df.sort_values(by='RSP', ascending=False).reset_index(drop=True)

    return df


In [6]:
# function to get stock technicals
def stock_prec_dev(stock_symbol):
    #stock_symbol = 'ULTRACEMCO.NS'
    short_window = 20
    mid_window = 50
    long_window = 200
    moving_avg = 'SMA'

    start = datetime.datetime(*map(int, start_date.split('-')))
    end = datetime.datetime(*map(int, end_date.split('-')))

    stock_df = yfin.Ticker(stock_symbol + '.NS').history(period='max', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

    stock_df['Max'] = round(max(stock_df['Close']),2)
    stock_df = stock_df[(stock_df.index <= end_date) & (stock_df.index >= start_date)]
    stock_df['200_SMA'] = round(stock_df['Close'].rolling(window = 200, min_periods = 1).mean(),0)
    stock_df['Dev%_200'] = round((stock_df['Close'] - stock_df['200_SMA'])*100/stock_df['200_SMA'],2)
    stock_df.dropna(axis = 0, inplace = True) # remove any null rows

    stock_df['20_SMA'] = stock_df['Close'].rolling(window=20).mean()
    stock_df['50_SMA'] = stock_df['Close'].rolling(window=50).mean()
    stock_df['Symbol'] = stock_symbol

    stock_df['Close'] = round(stock_df['Close'],2)
    stock_df['Min'] = round(min(stock_df['Close']),2)
    stock_df['RSI_14'] = ta.momentum.RSIIndicator(close=stock_df['Close'], window=14).rsi()
    stock_df['RSI_14'] = round(stock_df['RSI_14'],0)
    stock_df['Prev_Close'] = stock_df['Close'].shift(1)
    stock_df.drop(['Open', 'Low', 'High', 'Volume'], axis=1, inplace=True)
    stock_df = stock_df.tail(1)

    max_SMA = max(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    min_SMA = min(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    ABS_Spread = max_SMA - min_SMA
    stock_df['Spread%'] = round((ABS_Spread / stock_df['200_SMA'].item()) * 100,2)

    return stock_df

# function to compute stock attributes
def get_common_features(stock_ids, df_mypf):

  df_prec_dev = pd.DataFrame()
  df_tmp = get_screener_features()
  df_rs = get_relative_strength(stock_ids)

  for stock_id in stock_ids:
      tmp = stock_prec_dev(stock_id)
      tmp = tmp.reset_index()
      df_prec_dev = pd.concat([df_prec_dev, tmp], ignore_index = True)
  df_prec_dev = pd.merge(df_prec_dev, df_mypf, on= 'Symbol')
  df_prec_dev = pd.merge(df_prec_dev, df_tmp, on= 'Symbol', how='left')
  df_prec_dev = pd.merge(df_prec_dev, df_rs, on= 'Symbol', how='left')
  #print(df_prec_dev.columns)
  df_prec_dev['Curr_PE'] = round(df_prec_dev['Close']/df_prec_dev['EPS'],1)
  df_prec_dev['Dev%_PE'] = round((df_prec_dev['Curr_PE'] - df_prec_dev['MedPE'])*100/df_prec_dev['MedPE'],2)
  df_prec_dev['Conviction'] = df_prec_dev['Conviction'] + '-' + df_prec_dev['CapType']
  return df_prec_dev

# function to arrange stock features
def arrange_features(df_stocks, common_cols, diff_cols):
  df_stocks_common = df_stocks[common_cols].drop_duplicates()
  df_stocks_diff = df_stocks[diff_cols]
  df_stocks_diff['Investment'] = df_stocks_diff['AvgCost'] * df_stocks_diff['Shares']
  df_stocks_diff = df_stocks_diff.groupby(['Symbol'])[['Shares', 'Investment']].aggregate(['sum']).reset_index()
  df_stocks_diff.columns = ['Symbol', 'Shares', 'Investment']
  df_stocks_diff['AvgCost'] = round(df_stocks_diff['Investment']/df_stocks_diff['Shares'],2)
  df_stocks = pd.merge(df_stocks_diff, df_stocks_common, on='Symbol')
  return df_stocks

# plot fact distribution across dimension
def plot_pie_chart(df, dimension, fact):
  # grouping the data by category and calculating the sum of fact for each type
  grouped_data = df.groupby(dimension)[fact].sum()

  # sorting the grouped data in descending order
  grouped_data = grouped_data.sort_values(ascending=False)

  # creating a pie chart
  grouped_data.plot.pie(autopct='%1.1f%%', startangle=90, figsize=(6, 6))

  # adding a title and displaying the plot
  plt.title(f'{dimension} {fact} Distribution')
  plt.ylabel('')  # To hide the y-label
  plt.show()

In [7]:
# function to get portfolio features

def get_portfolio_features(df_common_features):

  print_date_time()
  print('=================== PORTFOLIO QUALITY ====================')

  #df_common_features["Target"] = df_common_features["Target"].fillna(df_common_features["Max"])
  df_common_features['NTT'] = np.where(df_common_features['Strategy']== 'NTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['LTT'] = np.where(df_common_features['Strategy']== 'BTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['BOL'] = df_common_features['Min']

  tmp_df = df_common_features[df_common_features['Symbol'].isin(mypf_df[mypf_df['InPortfolio'] != 'NA'].Symbol.values)]
  print('qualified stocks: '+str(len(tmp_df['Symbol'].unique())))
  tmp_df1 = tmp_df[tmp_df['LatestQtr'] == 1]
  print('with latest results: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df1 = tmp_df1[tmp_df1['StarStock'] == 1]
  print('still star stocks: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df['Investment'] = tmp_df['AvgCost'] * tmp_df['Shares']
  tmp_df['Current'] = round(tmp_df['Close'] * tmp_df['Shares'],0)
  tmp_df['Previous'] = tmp_df['Prev_Close'] * tmp_df['Shares']
  tmp_df['EstimatedST'] = tmp_df['NTT'] * tmp_df['Shares']
  tmp_df['EstimatedLT'] = tmp_df['LTT'] * tmp_df['Shares']
  tmp_df['Current P/L'] = round((tmp_df['Current'] - tmp_df['Investment']),0)
  tmp_df['Today P/L%'] = round((tmp_df['Current'] - tmp_df['Previous'])*100/tmp_df['Previous'],2)
  tmp_df['Current P/L%'] = round((tmp_df['Current'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedST P/L%'] = round((tmp_df['EstimatedST'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedLT P/L%'] = round((tmp_df['EstimatedLT'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['NTT%'] = round((tmp_df['NTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['LTT%'] = round((tmp_df['LTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['Gained%'] = round((tmp_df['Close'] - tmp_df['BOL'])*100/tmp_df['BOL'],2)
  investment = round(sum(tmp_df['AvgCost']*tmp_df['Shares']),0)
  current = round(sum(tmp_df['Close']*tmp_df['Shares']),0)
  tmp_df['InitAlloc%'] = round(tmp_df['Investment']*100/investment,2)
  tmp_df['CurrAlloc%'] = round(tmp_df['Current']*100/current,2)
  tmp_df['FTT'] = tmp_df['LTT']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT'] = tmp_df['NTT']
  tmp_df['FTT%'] = tmp_df['LTT%']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT%'] = tmp_df['NTT%']
  tmp_df['FTT Amt'] = round(tmp_df['FTT%'] * tmp_df['Current']/100,0)
  tmp_df['OTT%'] = round((tmp_df['FTT'] - tmp_df['AvgCost'])*100/tmp_df['AvgCost'],2)
  tmp_df['RRR Ind'] = round(tmp_df['Current P/L']/tmp_df['FTT Amt'],2)
  tmp_df['Risk Ind'] = round(tmp_df['Current P/L%']*tmp_df['CurrAlloc%'],0)

  return tmp_df

# function to print portfolio features
def print_portfolio_stats(df_portfolio_features, myinvmt_df):
  from pyxirr import xirr

  total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt = get_amt()

  dates = myinvmt_df['Date'].values
  dates = np.append(dates, datetime.date.today().strftime('%d-%b-%y'))
  investment = myinvmt_df['Investment'].values
  dates= pd.to_datetime(dates)

  current = round(sum(df_portfolio_features['Close']*df_portfolio_features['Shares']),0) + reserve_amt
  investment_xirr = np.append(investment, current)
  cagr = round(xirr(pd.DataFrame({"dates": dates, "amounts": investment_xirr}))*100,2)

  investment = -sum(investment)
  invested = round(sum(df_portfolio_features['AvgCost']*df_portfolio_features['Shares']),0) + reserve_amt
  previous = round(sum(df_portfolio_features['Prev_Close']*df_portfolio_features['Shares']),0) + reserve_amt
  cy_invested = investment + py_booked_amt

  today_pnl_amount = current-previous
  today_pnl_percentage = round((current-previous)*100/previous,2)

  curr_pnl_amount = current-invested
  curr_pnl_percentage = round((curr_pnl_amount)*100/(cy_invested),2)

  cy_pnl_amount = cy_booked_amt + curr_pnl_amount
  cy_pnl_percentage = round((cy_pnl_amount)*100/cy_invested,2)

  overall_pnl_amount = total_booked_amt + curr_pnl_amount
  overall_pnl_percentage = round((overall_pnl_amount)*100/investment,2)

  estimate_st = round(sum(df_portfolio_features['FTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_st_pnl_amount = estimate_st-current
  est_st_pnl_percentage = round((est_st_pnl_amount)*100/current,2)

  estimate_lt = round(sum(df_portfolio_features['LTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_lt_pnl_amount = estimate_lt-current
  est_lt_pnl_percentage = round((est_lt_pnl_amount)*100/current,2)

  total_profit = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Investment']),0)
  total_loss = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Investment']),0)

  cy_booked_percentage = round((cy_booked_amt)*100/current,2)
  py_booked_percentage = round((py_booked_amt)*100/investment,2)
  total_booked_percentage = round((total_booked_amt)*100/investment,2)

  print('=================== PORTFOLIO SNAPSHOT ===================')
  print('Deployed Value: ', format_amt(investment))
  print('Current Value:  ', format_amt(current))
  print('CAGR/XIRR %:    ', '{}%'.format(cagr))
  print('Est FTT Value:  ', format_amt(estimate_st))
  print('Future Upside:  ', '{} ({}%)'.format(
      format_amt(est_st_pnl_amount),
      est_st_pnl_percentage
  ))

  print('=================== PERFORMANCE ==========================')
  print('Current PnL:             {} ({}%)'.format(
      format_amt(curr_pnl_amount), curr_pnl_percentage
  ))
  print('Total Booked PnL:        {} ({}%)'.format(
      format_amt(total_booked_amt), total_booked_percentage
  ))
  print('Booked + Current PnL:    {} ({}%)'.format(
      format_amt(overall_pnl_amount), overall_pnl_percentage
  ))
  print('CY Booked PnL:           {} ({}%)'.format(
      format_amt(cy_booked_amt), cy_booked_percentage
  ))
  print('Prev Year Booked PnL:    {} ({}%)'.format(
      format_amt(py_booked_amt), py_booked_percentage
  ))

  print('=================== TODAY / OPERATIONS ===================')
  print('Today PnL:               {} ({}%)'.format(
      format_amt(today_pnl_amount), today_pnl_percentage
  ))
  print('CY Investment:           ', format_amt(cy_invested))
  print('Reserve:                 ', format_amt(reserve_amt))

  print('=================== P&L DETAIL ============================')
  print('Total Profit:            ', format_amt(total_profit))
  print('Total Loss:              ', format_amt(total_loss))
  print('============================================================')

In [8]:
# get start and end date
start_date, end_date = get_start_end_date()

# get portfolio and prospects data
mypfs_df = get_mypfs_df()
mypps_df = get_mypps_df()
myinvmt_df = get_myinvmt_df()

# merge above datasets
mypf_df = pd.merge(mypfs_df, mypps_df, on="Symbol")

# seggregate dm and sv portfolio
dm_pf = mypf_df[mypf_df['InPortfolio'] == 'DM']
sv_pf = mypf_df[mypf_df['InPortfolio'] == 'SV']

# build portfolio stock dataframe
dm_stocks = get_stock_ids(dm_pf)
sv_stocks = get_stock_ids(sv_pf)
df_stocks = pd.concat([dm_pf,sv_pf], ignore_index = True)

# arrange common and diff stock features
common_cols = ['Symbol', 'Target', 'Criteria', 'Strategy', 'CumlRnk', 'LatestQtr', 'StarStock', 'Conviction', 'Category', 'InFolio', 'XIRR', 'MBQ']
diff_cols = ['Symbol', 'AvgCost', 'Shares']
df_stocks = arrange_features(df_stocks, common_cols, diff_cols)

# get common features
stock_ids = df_stocks['Symbol'].values
df_common_features = get_common_features(stock_ids, df_stocks)

df_common_features.reset_index(drop=True, inplace=True)
df_common_features.drop(['Date'], axis=1, inplace=True)
# get and print portfolio features
df_portfolio_features = get_portfolio_features(df_common_features)

print_portfolio_stats(df_portfolio_features, myinvmt_df)

df = df_portfolio_features
#plot_pie_chart(df, 'CapType', 'Current')
list_ox40n = list(df[df['MBQ'].str.contains('OX40N', na=False)]['Symbol'].values)

Run date time (IST): 2026-11-09 19:56:14
=================== PORTFOLIO QUALITY ====================
qualified stocks: 84
with latest results: 84
still star stocks: 44
=================== PORTFOLIO SNAPSHOT ===================
Deployed Value:  1.49 C
Current Value:   1.53 C
CAGR/XIRR %:     1.11%
Est FTT Value:   2.74 C
Future Upside:   1.21 C (78.78%)
=================== PERFORMANCE ==========================
Current PnL:             -42.17 L (-21.97%)
Total Booked PnL:        49.23 L (33.02%)
Booked + Current PnL:    7.06 L (4.73%)
CY Booked PnL:           6.36 L (4.15%)
Prev Year Booked PnL:    42.87 L (28.76%)
=================== TODAY / OPERATIONS ===================
Today PnL:               1.05 L (0.69%)
CY Investment:            1.92 C
Reserve:                  2.04 K
=================== P&L DETAIL ============================
Total Profit:             2.05 L
Total Loss:               -44.22 L


In [9]:
# top 5 near their targets
cols = ['Symbol', 'Today P/L%', 'Current P/L%', 'FTT%', 'OTT%', 'FTT Amt', 'Current P/L', 'Current', 'FTT', 'Dev%_PE', 'RSI_14','Conviction', 'Spread%',  'CumlRnk', 'RRR Ind', 'CurrAlloc%', 'Gained%', 'Criteria', 'Strategy', 'Category']
df_tmp = df_portfolio_features[~df_portfolio_features['Symbol'].isin(['ENRIN','BLUSPRING','DIGITIDE'])]
df_tmp = df_tmp.sort_values(by = 'FTT Amt', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
75,TTKPRESTIG,-0.70,-21.42,27.38,0.09,21682.0,-21587.0,79190.0,770.00,-4.44,71.0,M-SC,2.66,250.0,-1.00,0.52,42.53,OX40N,NTT,DURABLES
12,BOSCH-HCIL,1.76,-24.59,32.60,-0.01,31419.0,-31429.0,96376.0,2282.00,752.74,47.0,M-SC,19.82,231.0,-1.00,0.63,64.05,OX40N,BTT,AC
65,SIS,2.38,-10.24,35.09,21.26,34973.0,-11365.0,99667.0,528.00,91.57,49.0,H-SC,7.44,168.0,-0.32,0.65,48.19,OX40N,NTT,MISC
76,UNITDSPR,1.02,5.22,14.77,20.77,36389.0,12229.0,246373.0,1644.00,-10.30,61.0,X-MC,3.10,64.0,0.34,1.61,18.43,X40N,NTT,BREWERIES
52,MASFIN,-0.58,-19.19,50.05,21.26,39629.0,-18801.0,79179.0,396.02,-37.37,50.0,H-SC,14.94,167.0,-0.47,0.52,7.88,XR,ATH,FINANCE


In [10]:
# top 5 to accumulate
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 40) & (df_tmp['FTT%'] > 20)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
38,HYUNDAI,-0.94,7.89,31.88,42.29,97537.0,22379.0,305950.0,2782.73,12.96,63.0,X-LC,4.53,29.0,0.23,2.00,25.94,X40N,ATH,AUTO
51,MARUTI,1.40,-4.86,32.43,25.99,104814.0,-16510.0,323200.0,17120.06,-11.28,73.0,X-LC,12.28,33.0,-0.16,2.11,15.74,XR,ATH,AUTO
47,JIOFIN,-0.52,-19.12,68.26,36.09,221681.0,-76785.0,324760.0,387.00,-38.00,64.0,X-LC,10.12,34.0,-0.35,2.12,12.47,XY24,BTT,FINANCE
4,ASIANPAINT,-1.44,-8.48,36.39,24.83,118707.0,-30208.0,326209.0,3422.40,-20.30,63.0,X-LC,5.43,30.0,-0.25,2.13,19.60,X40,ATH,PAINTS
8,BAJAJHLDNG,-0.92,-2.04,29.70,27.06,97493.0,-6827.0,328260.0,14191.91,-11.33,53.0,X-LC,3.76,28.0,-0.07,2.14,27.40,X40,ATH,FINANCE


In [11]:
# top 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
61,REPCOHOME,16.75,-16.58,127.54,89.82,304836.0,-47493.0,239012.0,880.00,NaN,76.0,NaN,13.27,161.0,-0.16,1.56,27.37,XY24,NTT,FINANCE
25,FINCABLES,12.96,56.09,32.26,106.44,74346.0,82815.0,230459.0,1629.94,13.95,49.0,M-SC,24.55,220.0,1.11,1.51,74.62,OX40N,ATH,CABLES
27,GLAXO,7.34,8.79,24.73,35.69,53615.0,17516.0,216800.0,3380.16,-18.41,61.0,X-MC,8.99,56.0,0.33,1.42,28.85,X40,ATH,PHARMA
63,SFL,5.12,-45.12,130.75,26.63,188149.0,-118318.0,143900.0,1287.00,NaN,44.0,NaN,7.55,242.0,-0.63,0.94,20.10,XY24,NTT,MISC
68,SYMPHONY,3.90,-59.26,145.37,-0.03,101359.0,-101416.0,69725.0,1306.00,NaN,52.0,NaN,25.57,205.0,-1.00,0.46,10.07,OX40N,NTT,DURABLES


In [12]:
# bottom 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
66,STARHEALTH,-2.92,6.09,38.81,47.26,115750.0,17125.0,298248.0,761.00,-6.49,52.0,H-SC,4.47,173.0,0.15,1.95,28.95,XY24,NTT,INSURANCE
79,VALIANTORG,-2.41,-28.94,145.07,74.15,168663.0,-47342.0,116263.0,838.00,NaN,34.0,NaN,41.92,145.0,-0.28,0.76,73.02,XR,NTT,CHEMICALS
18,COFFEEDAY,-2.38,-48.04,160.33,35.27,94598.0,-54547.0,59002.0,80.00,NaN,55.0,NaN,1.60,266.0,-0.58,0.39,45.50,XR,NTT,HOTELS
44,INFY,-2.31,-16.51,70.87,42.66,237194.0,-66166.0,334689.0,1972.00,-39.69,58.0,X-LC,8.18,3.0,-0.28,2.19,19.82,X40,NTT,IT
33,HCLTECH,-2.29,-3.72,30.57,25.70,106240.0,-13444.0,347529.0,1837.07,-8.41,58.0,X-LC,7.83,8.0,-0.13,2.27,38.66,X40,ATH,IT


In [13]:
# top 5 to exit based on CumlRnk
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > -1) & (df_tmp['Current P/L%'] < 1)].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category


In [14]:
# OX40N stocks to exit
df_tmp = df_portfolio_features[(df_portfolio_features['Symbol'].isin(list_ox40n))].sort_values(by = 'Current P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,12.96,56.09,32.26,106.44,74346.0,82815.0,230459.0,1629.94,13.95,49.0,M-SC,24.55,220.0,1.11,1.51,74.62,OX40N,ATH,CABLES
65,SIS,2.38,-10.24,35.09,21.26,34973.0,-11365.0,99667.0,528.00,91.57,49.0,H-SC,7.44,168.0,-0.32,0.65,48.19,OX40N,NTT,MISC
75,TTKPRESTIG,-0.70,-21.42,27.38,0.09,21682.0,-21587.0,79190.0,770.00,-4.44,71.0,M-SC,2.66,250.0,-1.00,0.52,42.53,OX40N,NTT,DURABLES
12,BOSCH-HCIL,1.76,-24.59,32.60,-0.01,31419.0,-31429.0,96376.0,2282.00,752.74,47.0,M-SC,19.82,231.0,-1.00,0.63,64.05,OX40N,BTT,AC
48,KANSAINER,2.54,-32.92,69.17,13.47,125116.0,-88785.0,180882.0,340.00,-29.72,62.0,H-SC,6.73,162.0,-0.71,1.18,27.85,XY24,NTT,PAINTS


In [15]:
# top 5 to book for rotation
df_tmp = df_portfolio_features[(df_portfolio_features['Current P/L%'] > 20) ].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,12.96,56.09,32.26,106.44,74346.0,82815.0,230459.0,1629.94,13.95,49.0,M-SC,24.55,220.0,1.11,1.51,74.62,OX40N,ATH,CABLES


In [16]:
# top 5 to monitor for exit
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > 0) ].sort_values(by = 'Dev%_PE', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,12.96,56.09,32.26,106.44,74346.0,82815.0,230459.0,1629.94,13.95,49.0,M-SC,24.55,220.0,1.11,1.51,74.62,OX40N,ATH,CABLES
66,STARHEALTH,-2.92,6.09,38.81,47.26,115750.0,17125.0,298248.0,761.00,-6.49,52.0,H-SC,4.47,173.0,0.15,1.95,28.95,XY24,NTT,INSURANCE


In [17]:
# top 5 to book for rotation from weak conviction
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[~df_tmp['Criteria'].isin(['XY25','XY24','X40', 'X40N'])].sort_values(by = 'RRR Ind', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,12.96,56.09,32.26,106.44,74346.0,82815.0,230459.0,1629.94,13.95,49.0,M-SC,24.55,220.0,1.11,1.51,74.62,OX40N,ATH,CABLES
83,WIPRO,0.51,-8.55,111.06,93.01,254161.0,-21390.0,228850.0,420.00,-22.62,69.0,M-LC,7.61,98.0,-0.08,1.50,26.92,XR,NTT,IT
69,TANLA,-0.44,-26.70,193.54,115.17,384123.0,-72289.0,198472.0,1903.93,NaN,73.0,NaN,15.48,147.0,-0.19,1.30,75.45,AR,ATH,IT
3,ALKYLAMINE,2.86,-31.10,179.46,92.54,124834.0,-31402.0,69561.0,4520.85,NaN,40.0,NaN,7.56,142.0,-0.25,0.45,32.09,SR,ATH,CHEMICALS
41,INDIAMART,-0.27,-33.70,203.04,100.92,166034.0,-41562.0,81774.0,4675.60,NaN,43.0,NaN,16.80,140.0,-0.25,0.53,3.75,AR,ATH,MISC


In [18]:
# top 5 to accumulate based on lowest RSI_14
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'RSI_14', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
37,HONAUT,0.07,-21.96,93.33,50.88,140462.0,-42338.0,150500.0,58192.80,-39.00,36.0,X-SC,7.30,90.0,-0.30,0.98,14.66,X40N,ATH,ELECTRICAL
6,AWL,-0.39,-42.72,191.55,67.02,390975.0,-152198.0,204111.0,485.00,-57.79,37.0,X-SC,7.83,81.0,-0.39,1.33,1.53,XY24,NTT,FMCG
11,BERGEPAINT,-1.49,-13.16,44.31,25.32,99175.0,-33929.0,223820.0,680.00,-24.45,49.0,X-MC,2.17,68.0,-0.34,1.46,17.37,XY24,NTT,PAINTS
20,DABUR,-0.42,-26.63,95.87,43.70,174480.0,-66072.0,181996.0,735.00,-36.80,50.0,X-MC,12.72,70.0,-0.38,1.19,5.85,XY24,BTT,FMCG
19,COLPAL,0.95,-35.64,114.04,37.76,193298.0,-93865.0,169500.0,3628.04,-22.06,51.0,X-MC,14.53,62.0,-0.49,1.11,7.79,XY25,ATH,FMCG


In [19]:
# top 5 to accumulate based on Spread and CumlRnk
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 100)].sort_values(by = 'Spread%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
17,CIPLA,-0.51,5.52,17.41,23.89,55235.0,16604.0,317259.0,1648.14,19.62,64.0,X-LC,0.81,18.0,0.30,2.07,18.83,X40N,ATH,PHARMA
1,ABBOTINDIA,1.27,-10.50,28.90,15.37,46263.0,-18775.0,160080.0,34390.26,-27.16,56.0,X-MC,1.03,58.0,-0.41,1.05,8.44,X40,ATH,PHARMA
54,PAGEIND,1.25,-5.11,40.01,32.85,101889.0,-13719.0,254660.0,50936.08,-29.48,56.0,X-MC,2.16,55.0,-0.13,1.66,21.60,X40,ATH,APPARELS
11,BERGEPAINT,-1.49,-13.16,44.31,25.32,99175.0,-33929.0,223820.0,680.00,-24.45,49.0,X-MC,2.17,68.0,-0.34,1.46,17.37,XY24,NTT,PAINTS
76,UNITDSPR,1.02,5.22,14.77,20.77,36389.0,12229.0,246373.0,1644.00,-10.30,61.0,X-MC,3.10,64.0,0.34,1.61,18.43,X40N,NTT,BREWERIES


In [20]:
# top 5 to accumulate based on lowest Gained%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'Gained%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
6,AWL,-0.39,-42.72,191.55,67.02,390975.0,-152198.0,204111.0,485.00,-57.79,37.0,X-SC,7.83,81.0,-0.39,1.33,1.53,XY24,NTT,FMCG
28,GODREJCP,-0.04,-22.62,70.71,32.10,136540.0,-56439.0,193098.0,1452.16,-24.07,51.0,X-MC,16.36,32.0,-0.41,1.26,4.62,X40N,ATH,FMCG
20,DABUR,-0.42,-26.63,95.87,43.70,174480.0,-66072.0,181996.0,735.00,-36.80,50.0,X-MC,12.72,70.0,-0.38,1.19,5.85,XY24,BTT,FMCG
26,GILLETTE,-0.74,-13.14,48.15,28.69,106192.0,-33351.0,220545.0,10891.01,-37.24,55.0,X-SC,6.17,83.0,-0.31,1.44,6.56,X40,ATH,FMCG
55,PGHH,2.23,-46.52,144.97,31.02,165556.0,-99320.0,114200.0,17484.41,-60.69,55.0,X-SC,23.80,57.0,-0.60,0.75,7.51,X40,ATH,FMCG


In [21]:
# top 5 to accumulate based on lowest CurrAlloc%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
55,PGHH,2.23,-46.52,144.97,31.02,165556.0,-99320.0,114200.0,17484.41,-60.69,55.0,X-SC,23.80,57.0,-0.60,0.75,7.51,X40,ATH,FMCG
2,ACC,-0.11,-44.73,216.30,74.81,320531.0,-119940.0,148188.0,3906.00,-45.40,62.0,X-SC,14.93,85.0,-0.37,0.97,9.78,XY24,BTT,CEMENT
37,HONAUT,0.07,-21.96,93.33,50.88,140462.0,-42338.0,150500.0,58192.80,-39.00,36.0,X-SC,7.30,90.0,-0.30,0.98,14.66,X40N,ATH,ELECTRICAL
1,ABBOTINDIA,1.27,-10.50,28.90,15.37,46263.0,-18775.0,160080.0,34390.26,-27.16,56.0,X-MC,1.03,58.0,-0.41,1.05,8.44,X40,ATH,PHARMA
81,VOLTAS,-0.81,-14.72,70.26,45.20,114886.0,-28227.0,163515.0,1856.00,17.49,57.0,X-MC,19.85,79.0,-0.25,1.07,15.21,XY25,NTT,AC


In [22]:
# top 5 to accumulate based on CumlRnk
df_tmp = df_portfolio_features.sort_values(by = 'CumlRnk', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
46,ITC,-1.54,-20.61,62.12,28.70,215796.0,-90210.0,347385.0,452.00,-28.31,62.0,X-LC,7.42,1.0,-0.42,2.27,9.33,X40,NTT,FMCG
71,TCS,-0.92,-25.45,70.01,26.75,254755.0,-124191.0,363884.0,4208.34,-39.62,60.0,X-LC,6.94,2.0,-0.49,2.38,26.21,X40,ATH,IT
44,INFY,-2.31,-16.51,70.87,42.66,237194.0,-66166.0,334689.0,1972.00,-39.69,58.0,X-LC,8.18,3.0,-0.28,2.19,19.82,X40,NTT,IT
73,TMPV,1.03,-26.45,103.39,49.60,362646.0,-126117.0,350755.0,600.00,-85.99,62.0,X-LC,16.75,4.0,-0.35,2.29,12.94,XY24,NTT,AUTO
33,HCLTECH,-2.29,-3.72,30.57,25.70,106240.0,-13444.0,347529.0,1837.07,-8.41,58.0,X-LC,7.83,8.0,-0.13,2.27,38.66,X40,ATH,IT


In [23]:
# top 5 for average up
df_tmp = df_portfolio_features[(df_portfolio_features['Dev%_200'] > 5)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
75,TTKPRESTIG,-0.70,-21.42,27.38,0.09,21682.0,-21587.0,79190.0,770.00,-4.44,71.0,M-SC,2.66,250.0,-1.00,0.52,42.53,OX40N,NTT,DURABLES
12,BOSCH-HCIL,1.76,-24.59,32.60,-0.01,31419.0,-31429.0,96376.0,2282.00,752.74,47.0,M-SC,19.82,231.0,-1.00,0.63,64.05,OX40N,BTT,AC
79,VALIANTORG,-2.41,-28.94,145.07,74.15,168663.0,-47342.0,116263.0,838.00,NaN,34.0,NaN,41.92,145.0,-0.28,0.76,73.02,XR,NTT,CHEMICALS
0,5PAISA,3.63,-33.10,68.75,12.90,100500.0,-72314.0,146182.0,593.00,-6.12,60.0,H-SC,0.57,170.0,-0.72,0.96,41.07,OX40N,NTT,FINANCE
69,TANLA,-0.44,-26.70,193.54,115.17,384123.0,-72289.0,198472.0,1903.93,NaN,73.0,NaN,15.48,147.0,-0.19,1.30,75.45,AR,ATH,IT


In [24]:
# top 5 RSP
df_tmp = df_portfolio_features.sort_values(by = 'RSP', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
69,TANLA,-0.44,-26.70,193.54,115.17,384123.0,-72289.0,198472.0,1903.93,NaN,73.0,NaN,15.48,147.0,-0.19,1.30,75.45,AR,ATH,IT
70,TATAELXSI,0.91,-43.94,135.51,32.03,100153.0,-57927.0,73908.0,9161.00,-36.36,65.0,H-SC,13.26,158.0,-0.58,0.48,29.84,OX40N,NTT,IT
83,WIPRO,0.51,-8.55,111.06,93.01,254161.0,-21390.0,228850.0,420.00,-22.62,69.0,M-LC,7.61,98.0,-0.08,1.50,26.92,XR,NTT,IT
50,M&M,0.10,2.74,14.35,17.48,44387.0,8241.0,309316.0,3762.88,-10.19,76.0,X-LC,7.59,25.0,0.19,2.02,19.61,X40N,ATH,AUTO
13,BSOFT,-1.50,-39.76,144.33,47.20,122299.0,-55917.0,84736.0,811.90,NaN,61.0,NaN,10.81,172.0,-0.46,0.55,25.06,XR,ATH,IT


In [25]:
# Top N allocation
df_tmp = df_portfolio_features.sort_values(by = 'CurrAlloc%', ascending=False)
top_n_values = [10, 25, 50]

sum_df = pd.DataFrame({
    'Top_N': top_n_values,
    'Sum_Alloc%': [df_tmp['CurrAlloc%'].head(n).sum() for n in top_n_values]
})

sum_df

,Top_N,Sum_Alloc%
0,10,22.79
1,25,51.66
2,50,82.32


In [26]:
# market-cap-wise allocation
df_tmp = df_portfolio_features[cols]
df_tmp.groupby(df_tmp['Conviction'].str[-2:])['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
LC    43.23
SC    20.31
MC    19.83
Name: CurrAlloc%, dtype: float64

In [27]:
# criteria-wise allocation
df_portfolio_features.groupby('Criteria')['CurrAlloc%'].sum().sort_values(ascending=False)

Criteria
X40      27.37
XY24     22.85
X40N     19.18
XR        9.99
XY25      9.94
OX40N     6.75
AR        3.19
SR        0.75
Name: CurrAlloc%, dtype: float64

In [28]:
# conviction-wise allocation
df_portfolio_features.groupby('Conviction')['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
X-LC    41.73
X-MC    17.42
H-SC     8.19
X-SC     7.00
M-SC     5.12
M-LC     1.50
M-MC     1.10
H-MC     0.98
L-MC     0.33
Name: CurrAlloc%, dtype: float64

In [29]:
# sector-wise stats
df_tmp = df_portfolio_features.groupby('Category')[['CurrAlloc%', 'Current', 'Current P/L', 'FTT Amt']].sum().sort_values(by=['Current', 'Current P/L'], ascending=False)
df_tmp['Current P/L%'] = round(df_tmp['Current P/L'] * 100 / df_tmp['Current'], 2)
df_tmp['FTT%'] = round(df_tmp['FTT Amt'] * 100 / df_tmp['Current'], 2)
cols = ['CurrAlloc%', 'Current P/L%', 'FTT%']
df_tmp[cols].sort_values(by=['CurrAlloc%'], ascending=False)

,CurrAlloc%,Current P/L%,FTT%
Category,,,
FMCG,18.35,-25.46,70.71
IT,12.86,-30.42,98.37
FINANCE,11.03,-20.76,72.15
AUTO,9.31,-12.15,53.35
PAINTS,5.51,-25.35,47.86
MISC,5.48,-46.75,105.74
INSURANCE,5.31,-1.66,39.86
BANKS,5.15,-13.85,70.94
PHARMA,4.54,2.21,22.35


In [30]:
# money to be made criteria-wise
df_portfolio_features.groupby('Criteria')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Criteria,,
XY24,3996686.0,19
X40,2145258.0,16
XR,1576669.0,13
X40N,1495962.0,12
AR,1019490.0,6
XY25,769051.0,6
OX40N,758253.0,10
SR,298615.0,2


In [31]:
# money to be made conviction-wise
df_portfolio_features.groupby('Conviction')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Conviction,,
X-LC,2947489.0,19
X-MC,1569722.0,13
X-SC,1202545.0,6
H-SC,898378.0,8
M-SC,380310.0,6
H-MC,263751.0,1
M-LC,254161.0,1
M-MC,77471.0,1
L-MC,51586.0,1


In [32]:
# money to be made criteria and conviction-wise
df_portfolio_features.groupby(['Conviction', 'Criteria'])['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

sum  count
Conviction Criteria                  
X-LC       X40       1302339.0      8
X-SC       XY24       711506.0      2
X-LC       XY24       584327.0      2
           X40N       572613.0      5
H-SC       XY24       550947.0      3
X-MC       X40N       537621.0      4
           X40        450262.0      5
X-LC       XY25       383396.0      3
X-MC       XY25       308184.0      2
           XY24       273655.0      2
X-SC       X40        271748.0      2
H-MC       XY24       263751.0      1
M-LC       XR         254161.0      1
H-SC       OX40N      235626.0      3
X-SC       X40N       219291.0      2
M-SC       OX40N      188563.0      4
           XY24       131404.0      1
H-SC       XR         111805.0      2
X-LC       XR         104814.0      1
M-MC       XY25        77471.0      1
M-SC       XR          60343.0      1
L-MC       XR          51586.0      1

In [33]:
# notebook execution time

end_time = time.time()
execution_time = round(end_time - start_time, 0)
print(f"Notebook execution time: {execution_time} seconds")

Notebook execution time: 35.0 seconds
